# EDS Week 02 — Student Edition

Missing values and outlier detection

Complete the 10 numbered blanks (`____`) as we work through the lesson. Replace each placeholder before running its cell. Run cells in order in Google Colab; incomplete cells will raise an error. Use the hints and discuss the results before moving on.

## Setup

In [ ]:
import os
os.chdir("/content")

In [ ]:
!rm -rf /content/Data
!git clone https://github.com/KU-EBL/Data.git

In [ ]:
import os
os.chdir("/content/Data/EDS")
os.listdir()

## 1. Inspect missing values

In [ ]:
import pandas as pd
df = pd.read_csv('01_few_missing_row_removal.csv')

In [ ]:
df

In [ ]:
df.isna()

In [ ]:
df.isna().sum()

### Checkpoint 01

Calculate the fraction of missing values from a Boolean mask. Enter a method name.

In [ ]:
missing_ratio = df.isna().____() * 100
missing_ratio

## 2. Remove incomplete rows

In [ ]:
import pandas as pd
df = pd.read_csv('01_few_missing_row_removal.csv')
print("Before: ", df.shape)
df

### Checkpoint 02

Remove rows containing missing values. Enter a method name.

In [ ]:
df_clean = df.____()
print("After: ", df_clean.shape)
df_clean

## 3. Impute numerical values

In [ ]:
df = pd.read_csv('02_numeric_mean_median_imputation.csv')
df

### Checkpoint 03

Replace missing soil moisture values with the arithmetic average. Enter a method name.

In [ ]:
df = pd.read_csv('02_numeric_mean_median_imputation.csv')
df["soil_moisture_pct"] = df["soil_moisture_pct"].fillna(df["soil_moisture_pct"].____())
df

### Checkpoint 04

Replace missing soil moisture values with the middle value. Enter a method name.

In [ ]:
df = pd.read_csv('02_numeric_mean_median_imputation.csv')
df["soil_moisture_pct"] = df["soil_moisture_pct"].fillna(df["soil_moisture_pct"].____())
df

## 4. Impute categorical values

In [ ]:
df = pd.read_csv('03_categorical_mode_unknown.csv')
df

In [ ]:
mode_value = df["land_use"].mode()
print(mode_value)
print(mode_value[0]) # df["land_use"].mode()[0]

In [ ]:
df["land_use"] = df["land_use"].fillna(mode_value[0])
df

In [ ]:
df = pd.read_csv('03_categorical_mode_unknown.csv')
df["land_use"] = df["land_use"].fillna("Unknown")
df

In [ ]:
df = pd.read_csv('03_categorical_mode_unknown.csv')

# 1. Select rows with both land use type and temperature available
known = df.dropna(subset=["land_use", "temperature_C"])

# 2. Select rows with missing land use type but available temperature
missing = df.index[
    df["land_use"].isna() & df["temperature_C"].notna()]

# 3. Fill missing land use types using the row with the closest temperature
for idx in missing:
    difference = (
        known["temperature_C"] - df.loc[idx, "temperature_C"]
    ).abs()

    nearest_idx = difference.idxmin()
    df.loc[idx, "land_use"] = known.loc[nearest_idx, "land_use"]

df

## 5. Fill gaps in time series

In [ ]:
!pip install -q pykalman
import numpy as np
from pykalman import KalmanFilter

df = pd.read_csv("04_timeseries_interpolation_kalman.csv", parse_dates = ["datetime"])
df

### Checkpoint 05

Estimate missing values between observed values using linear interpolation. Enter a method name.

In [ ]:
df_int = df.copy()
df_int["temperature_C"] = df_int["temperature_C"].____()
df_int

In [ ]:
df_kal = df.copy()
values = df_kal["temperature_C"].to_numpy(dtype=float)
masked_values = np.ma.masked_invalid(values)

kf = KalmanFilter(
    initial_state_mean = np.nanmean(values),
    initial_state_covariance=1.0,
    observation_covariance = 1.0,
    transition_covariance = 0.1
)

state_means, _ = kf.smooth(masked_values)
kalman_values = state_means.ravel()

missing_mask = df_kal["temperature_C"].isna()

df_kal.loc[missing_mask, "temperature_C"] = kalman_values[missing_mask]

df_kal

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# 1. Original data
axes[0].plot(
    df["datetime"],
    df["temperature_C"],
    marker="o"
)

axes[0].set_title("Original Data")
axes[0].set_xlabel("Datetime")
axes[0].set_ylabel("Temperature (°C)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(True)

# 2. Interpolation
axes[1].plot(
    df_int["datetime"],
    df_int["temperature_C"],
    marker="o"
)

axes[1].set_title("Linear Interpolation")
axes[1].set_xlabel("Datetime")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(True)

# 3. Kalman smoothing
axes[2].plot(
    df_kal["datetime"],
    df_kal["temperature_C"],
    marker="o"
)

axes[2].set_title("Kalman Smoothing")
axes[2].set_xlabel("Datetime")
axes[2].tick_params(axis="x", rotation=45)
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 6. Groupwise imputation

In [ ]:
df = pd.read_csv("05_groupwise_imputation.csv")
df

### Checkpoint 06

Calculate a separate mean for each land use category. Enter the grouping column name; keep the quotes.

In [ ]:
df_grp = df.copy()

group_mean = df_grp.groupby("____")["soil_moisture_pct"].transform("mean")

df_grp["soil_moisture_pct"] = df_grp["soil_moisture_pct"].fillna(group_mean)

df_grp

## 7. Remove variables with high missingness

In [ ]:
df = pd.read_csv("06_high_missingness_remove_variable.csv")
df

### Checkpoint 07

Remove columns with more than 50% missing values. Enter the percentage threshold.

In [ ]:
# Calculate the percentage of missing values in each column
missing_ratio = df.isna().mean() * 100
print(missing_ratio)

# Drop columns with more than 50% missing values
threshold = ____
cols_to_drop = missing_ratio[missing_ratio > threshold].index
df_reduced = df.drop(columns=cols_to_drop)

print("Dropped columns:", list(cols_to_drop))
df_reduced

## 8. Rule-based outlier detection

In [ ]:
df = pd.read_csv("07_outlier_detection_practice.csv")
df

In [ ]:
rule_temp_outliers = df[
    (df["temperature_C"] < -5) |
    (df["temperature_C"] > 50)
]

rule_temp_outliers

In [ ]:
df1 = df.copy()
df1 = df1.drop(index=rule_temp_outliers.index)

df1

## 9. IQR-based outlier detection

### Checkpoint 08

Calculate the spread of the middle 50% of the observations. Enter an expression.

In [ ]:
# Load the data
df = pd.read_csv("07_outlier_detection_practice.csv")

# Calculate the first and third quartiles
Q1 = df["nitrate_mg_L"].quantile(0.25)
Q3 = df["nitrate_mg_L"].quantile(0.75)

# Calculate the interquartile range (IQR)
IQR = ____

# Define lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

# 5. Identify nitrate values outside the bounds
iqr_outliers = df[
    (df["nitrate_mg_L"] < lower_bound) |
    (df["nitrate_mg_L"] > upper_bound)
]

iqr_outliers

In [ ]:
# Remove rows containing the identified nitrate outliers
df_iqr = df.drop(index=iqr_outliers.index)

df_iqr

## 10. Z-score outlier detection

In [ ]:
import pandas as pd
from scipy.stats import zscore

# Load the data
df = pd.read_csv("07_outlier_detection_practice.csv")

# Calculate z-scores for temperature
df["temperature_zscore"] = zscore(df["temperature_C"])

# Display temperature values and their z-scores
df[["site_id", "temperature_C", "temperature_zscore"]]

### Checkpoint 09

Detect unusually high and unusually low values using z-scores. Enter a method name.

In [ ]:
# Identify temperature outliers with an absolute z-score greater than 3
zscore_outliers = df[
    df["temperature_zscore"].____() > 3
]

zscore_outliers

## 11. Modified z-score outlier detection

In [ ]:
import numpy as np

x = df["temperature_C"]

# Calculate the median
median = x.median()

# Calculate the median absolute deviation (MAD)
MAD = np.median(np.abs(x - median))

# Calculate modified z-scores
df["modified_zscore"] = 0.6745 * (x - median) / MAD

# Identify potential outliers with an absolute modified z-score greater than 3.5
modified_z_outliers = df[
    df["modified_zscore"].abs() > 3.5
]

modified_z_outliers

In [ ]:
df_modified = df.drop(index=modified_z_outliers.index)

df_modified

## 12. Percentile-based screening

In [ ]:
lower_percentile = df["nitrate_mg_L"].quantile(0.05)
upper_percentile = df["nitrate_mg_L"].quantile(0.95)

print("Lower threshold:", lower_percentile)
print("Upper threshold:", upper_percentile)

In [ ]:
percentile_outliers = df[
    (df["nitrate_mg_L"] < lower_percentile) |
    (df["nitrate_mg_L"] > upper_percentile)
]

percentile_outliers

## 13. Visual inspection

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

plt.boxplot(df["nitrate_mg_L"])

plt.ylabel("Nitrate (mg/L)")
plt.title("Boxplot of Nitrate Concentration")

plt.show()

## 14. Multivariate outlier detection

In [ ]:
from sklearn.ensemble import IsolationForest

df_iso = df.copy()

# Select variables for multivariate outlier detection
features = [
    "temperature_C",
    "pH",
    "nitrate_mg_L",
    "DO_mg_L"
]

X = df_iso[features]

In [ ]:
# Fit the Isolation Forest model and predict outlier labels
model = IsolationForest(
    contamination=0.15,
    random_state=42
)

# Labels: 1 = inlier, -1 = outlier
df_iso["isolation_result"] = model.fit_predict(X)

df_iso

### Checkpoint 10

Select the label assigned to outliers by Isolation Forest. Enter the label.

In [ ]:
# Select rows identified as outliers
isolation_outliers = df_iso[
    df_iso["isolation_result"] == ____
]

isolation_outliers